# 03 — Grad-CAM Visualisation

Visualize which regions of an X-ray the model focuses on.

**Covers**: single-image Grad-CAM, multi-class grid, batch visualisation.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image
import torch

from src.models.model import load_model_for_inference
from src.data.transforms import get_gradcam_transform
from src.explainability.gradcam import GradCAM
from src.explainability.visualise import (
    generate_gradcam_overlay,
    generate_multi_class_grid,
    get_top_predictions
)
from src.utils.config import Paths, PATHOLOGY_CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = load_model_for_inference(Paths.best_model, device)
print('Model loaded')

## 2. Single-Image Grad-CAM

In [ ]:
# Load a sample X-ray
xray_path = next(Paths.images_dir().glob('*.png'), None)
if xray_path is None:
    print('No images found — run prepare_data() first')
else:
    image  = Image.open(xray_path).convert('RGB')
    tensor = get_gradcam_transform()(image)

    # Get predictions
    probs    = model.predict_single(tensor, device)
    top_preds = get_top_predictions(probs, threshold=0.1)
    top_cls  = top_preds[0]['class_name'] if top_preds else PATHOLOGY_CLASSES[0]

    # Generate Grad-CAM for top class
    with GradCAM(model, model.get_features_layer()) as cam:
        heatmap = cam.generate(tensor, top_cls, device)

    overlay = generate_gradcam_overlay(image, heatmap)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original X-ray', fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(f'Grad-CAM: {top_cls}  (p={probs[top_cls]:.2f})', fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Top predictions: {[(p["class_name"], f"{p["probability"]:.2f}") for p in top_preds[:3]]}')

## 3. Multi-class Grad-CAM Grid

In [ ]:
if xray_path:
    with GradCAM(model, model.get_features_layer()) as cam:
        all_heatmaps = cam.generate_all_classes(tensor, device)

    grid = generate_multi_class_grid(image, all_heatmaps, probs, top_k=4)

    plt.figure(figsize=(16, 4))
    plt.imshow(grid)
    plt.axis('off')
    plt.title('Top-4 Grad-CAM Heatmaps', fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. Save Sample Grad-CAMs for Model Card

In [ ]:
from src.explainability.visualise import save_gradcam_sample

if xray_path:
    for pred in top_preds[:3]:
        path = save_gradcam_sample(
            original_image = image,
            heatmap        = all_heatmaps[pred['class_name']],
            class_name     = pred['class_name'],
            probability    = pred['probability'],
        )
        print(f'Saved: {path}')